In [1]:
from glob import glob
from src.utils.schema_loader import DomainSchema
from src.hybrid_extractor import HybridEntityExtractor
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from src.utils.constants import *
from src.entities.multimodal_grounding import MultimodalGrounderOpenCLIP
from src.neo4j_manager import Neo4jManager

c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = OllamaLLM(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_URI,
    temperature=0
)

In [3]:
schema = DomainSchema(schema_file="./configs/cybersecurity.yml")
extractor = HybridEntityExtractor(
    config={
        "spacy_model": "en_core_web_lg",
        "use_llm_entities": True,
        "use_llm_relations": True
    },
    domain_schema=schema,
    llm=llm
)
text = "GinWui backdoor was created by Wicked Rose."
entities = extractor.extract_entities(text)
relations = extractor.extract_relations(text, entities)
for e in entities:
    print(vars(e))
for r in relations:
    print(vars(r))

c:\Users\ishani.kathuria\projects\TrustworthyRAG\src\hybrid_extractor.py:78: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  llm_response = self.llm(prompt)
2025-11-03 12:40:15,784 - HybridEntityExtractor - INFO - LLM response:
[
  {
    "text": "GinWui backdoor",
    "label": "MALWARE",
    "start_pos": 0,
    "end_pos": 12
  },
  {
    "text": "Wicked Rose",
    "label": "PERSON",
    "start_pos": 23,
    "end_pos": 33
  }
]
2025-11-03 12:40:16,187 - HybridEntityExtractor - INFO - LLM relation response:
[
  {
    "head": "GinWui backdoor",
    "tail": "Wicked Rose",
    "type": "CREATED_BY"
  }
]


{'text': 'GinWui', 'label': 'ORGANIZATION', 'start_pos': 0, 'end_pos': 6, 'confidence': 0.85, 'source_doc': '', 'metadata': {}}
{'text': 'Wicked Rose', 'label': 'PERSON', 'start_pos': 31, 'end_pos': 42, 'confidence': 0.85, 'source_doc': '', 'metadata': {}}
{'text': 'backdoor', 'label': 'MALWARE', 'start_pos': 7, 'end_pos': 15, 'confidence': 0.95, 'source_doc': '', 'metadata': {'from': 'pattern'}}
{'text': 'GinWui', 'label': 'MALWARE', 'start_pos': 0, 'end_pos': 6, 'confidence': 0.95, 'source_doc': '', 'metadata': {'from': 'pattern'}}
{'text': 'GinWui backdoor', 'label': 'MALWARE', 'start_pos': 0, 'end_pos': 12, 'confidence': 0.8, 'source_doc': '', 'metadata': {'from': 'llm'}}
{'head_entity': Entity(text='backdoor', label='MALWARE', start_pos=7, end_pos=15, confidence=0.95, source_doc='', metadata={'from': 'pattern'}), 'tail_entity': Entity(text='Wicked Rose', label='PERSON', start_pos=31, end_pos=42, confidence=0.85, source_doc='', metadata={}), 'relation_type': 'CREATED_BY', 'confiden

In [4]:
neo4j_manager = Neo4jManager(
    uri=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

2025-11-03 12:40:16,229 - Neo4jManager - INFO - Connected to Neo4j at neo4j://127.0.0.1:7687
2025-11-03 12:40:16,244 - Neo4jManager - INFO - Neo4j schema initialized


In [5]:
image_paths = glob("./data/extracted_images/*")[:2]

# Map image paths to their captions extracted from documents
caption_map = {
    './data/extracted_images\\WickedRose_andNCPH-6_page1_0_Wicked Roses Website wwwmghackercom.png': 'NCHP 5.0 Screenshot (GinWui Rootkit)',
    './data/extracted_images\\WickedRose_andNCPH-6_page1_1_NCPH Studio website wwwncphnet.png': 'RipGof attacks reveal a Chinese string related to systematic testing of offsets for exploitation.'
	}

In [6]:
full_text = "GinWui backdoor was created by Wicked Rose. " + \
			"NCHP 5.0 Screenshot (GinWui Rootkit). " + \
			"RipGof attacks reveal a Chinese string related to systematic testing of offsets for exploitation."
grounder = MultimodalGrounderOpenCLIP(neo4j_manager=neo4j_manager)
grounder.process(entities, image_paths, caption_map, full_text)

Encriched texts for embedding: ['GinWui backdoor was created by Wicked Rose. NCHP 5.0 Screenshot (GinWui Rootkit). RipGof attacks reveal a ', 'GinWui backdoor was created by Wicked Rose. NCHP 5.0 Screenshot (GinWui Rootkit). RipGof attacks reveal a Chinese string related to systematic', 'GinWui backdoor was created by Wicked Rose. NCHP 5.0 Screenshot (GinWui Rootkit). RipGof attacks reveal a Chinese s', 'GinWui backdoor was created by Wicked Rose. NCHP 5.0 Screenshot (GinWui Rootkit). RipGof attacks reveal a ', 'GinWui backdooroor was created by Wicked Rose. NCHP 5.0 Screenshot (GinWui Rootkit). RipGof attacks reveal a Chines']
Found 0 entity-image links with similarity > 0.75
